# Adaptive NeSy-Gen: IU-Xray and MIMIC-CXR experiments

This notebook uses the existing Drive manifests, PrimeKG caches, and MedSigLIP indices. It supports frozen-vision CheXagent QLoRA and MedGemma with no task-specific fine-tuning, followed by adaptive claim verification and replayed ablations. Use a GPU runtime.

**Environment boundary:** CheXagent's released code pins `transformers==4.40.0`; MedGemma needs a newer Transformers release. Run those sections in separate Colab runtimes and save outputs to Drive. Do not install both extras in one runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DATASET = 'iuxray_official'  # 'iuxray_official' or 'mimic_aug'
REPO_URL = 'https://github.com/FaezehMillerAI/NesyGENAAAI.git'
REPO_DIR = '/content/adaptive-nesy-gen'
OUTPUT_ROOT = f'/content/drive/MyDrive/aaai_2026_experiments/adaptive_nesy_gen/{RUN_DATASET}'


In [ ]:
import os, pathlib, subprocess
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
print('Repository:', pathlib.Path.cwd())


In [ ]:
import json, numpy as np
from pathlib import Path
from adaptive_nesy_gen.retrieval import load_manifest

PATHS = json.loads(Path('configs/colab_paths.json').read_text())[RUN_DATASET]
MANIFEST = PATHS['manifest']
RAW_PRIMEKG_DIR = Path('/content/drive/MyDrive/dataverse_files')
RAD_PRIMEKG_DIR = Path(f'/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}')
PRIMEKG_CACHE = str(RAD_PRIMEKG_DIR)
MEDSIGLIP_CACHE = PATHS['medsiglip_cache']
REBUILD_RAD_PRIMEKG = False
if REBUILD_RAD_PRIMEKG:
    subprocess.run([
        'python', 'scripts/build_radiology_primekg.py',
        '--primekg-dir', str(RAW_PRIMEKG_DIR), '--manifest', MANIFEST,
        '--output-dir', str(RAD_PRIMEKG_DIR), '--hops', '1',
        '--seed-split', 'train',
    ], check=True)
studies = load_manifest(MANIFEST)
counts = {split: sum(s.split == split for s in studies) for split in ('train', 'val', 'test')}
missing = [s.image_path for s in studies if not Path(s.image_path).exists()]
archive = np.load(MEDSIGLIP_CACHE, allow_pickle=False)
embedding_key = next(k for k in ('embeddings', 'image_embeddings', 'features') if k in archive)
assert len(archive[embedding_key]) == counts['train'], (len(archive[embedding_key]), counts)
assert counts['train'] and counts['val'] and counts['test']
assert not missing, f'{len(missing)} manifest images are missing; first={missing[:3]}'
assert RAD_PRIMEKG_DIR == Path(PATHS['primekg_cache'])
for required in ('kg.csv', 'nodes.csv', 'radiology_primekg_summary.json'):
    assert (RAD_PRIMEKG_DIR / required).exists(), RAD_PRIMEKG_DIR / required
print({'splits': counts, 'index_shape': archive[embedding_key].shape, 'primekg': PRIMEKG_CACHE})


In [ ]:
# Fast contract checks before any expensive model load.
subprocess.run(['pytest', '-q'], check=True)
subprocess.run(['adaptive-nesy-gen', 'demo', '--output', '/content/demo_trace.json'], check=True)


## A. Efficient CheXagent QLoRA (dedicated runtime)

The script uses NF4 QLoRA only on decoder modules, keeps the entire visual encoder frozen in BF16, applies 512×512 augmentation without flips, masks prompt tokens from the loss, uses only train/val, and inserts the first three of five unique MedSigLIP neighbours into 50% of training prompts. Set `RUN_QLORA=True` when ready.

In [ ]:
RUN_QLORA = False
MAX_STEPS = 500  # 20 for a pipeline check; 500+ for the planned run
CHEX_OUTPUT = f'{OUTPUT_ROOT}/chexagent_qlora'
if RUN_QLORA:
    subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[chexagent]'], check=True)
    cmd = [
        'python', 'scripts/train_chexagent_qlora.py',
        '--manifest', MANIFEST, '--medsiglip-cache', MEDSIGLIP_CACHE,
        '--output-dir', CHEX_OUTPUT, '--max-steps', str(MAX_STEPS),
    ]
    subprocess.run(cmd, check=True)
else:
    print('QLoRA skipped. Set RUN_QLORA=True to train.')


## B. Generate a test subset once

Choose one backend per fresh runtime. For CheXagent install `.[chexagent]` and provide its adapter. For MedGemma install `.[medgemma]`, accept the gated Hugging Face terms, and authenticate. On MIMIC-CXR describe MedGemma as **no task-specific fine-tuning**, not strict unseen-data zero-shot. Start with `LIMIT=20`, then remove the limit for the full run.

In [ ]:
BACKEND = 'medgemma'  # 'medgemma', 'chexagent', or 'retrieval' baseline
DRAFTING_MODE = 'few-shot'  # 'zero-shot' or 'few-shot'
LIMIT = 20
if BACKEND == 'medgemma':
    subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[medgemma]'], check=True)
    from huggingface_hub import login
    login()
elif BACKEND == 'chexagent':
    subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[chexagent]'], check=True)
else:
    subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[medgemma]'], check=True)

draft_file = f'{OUTPUT_ROOT}/{BACKEND}_{DRAFTING_MODE}_full_adaptive.jsonl'
cmd = [
    'python', 'scripts/run_experiments.py', '--manifest', MANIFEST,
    '--medsiglip-cache', MEDSIGLIP_CACHE, '--primekg-cache', PRIMEKG_CACHE,
    '--output', draft_file, '--backend', BACKEND, '--drafting-mode', DRAFTING_MODE,
    '--ablation', 'full_adaptive', '--top-k', '5',
]
if LIMIT: cmd += ['--limit', str(LIMIT)]
if BACKEND == 'chexagent': cmd += ['--adapter', f'{CHEX_OUTPUT}/adapter']
subprocess.run(cmd, check=True)
print(draft_file)


## C. Replay verifier ablations

Generation is the expensive and stochastic comparison unit. These runs reuse each saved raw draft and rerun only retrieval, linking, graph constraints, gate, and revision. Add report-level verification and shuffled/relation-ablated PrimeKG caches as separate controls.

In [ ]:
suite_dir = f'{OUTPUT_ROOT}/replay_{BACKEND}_{DRAFTING_MODE}_suite'
cmd = [
    'python', 'scripts/run_experiments.py', '--manifest', MANIFEST,
    '--medsiglip-cache', MEDSIGLIP_CACHE, '--primekg-cache', PRIMEKG_CACHE,
    '--output', suite_dir, '--backend', 'replay', '--drafts-jsonl', draft_file,
    '--drafting-mode', DRAFTING_MODE, '--suite', '--top-k', '5',
]
if LIMIT: cmd += ['--limit', str(LIMIT)]
subprocess.run(cmd, check=True)
ablation_files = {p.stem: str(p) for p in Path(suite_dir).glob('*.jsonl')}
print('Completed:', sorted(ablation_files))


In [ ]:
# Lightweight explanation/latency summary. Use official CheXbert/CheXpert,
# RadGraph, METEOR and CIDEr implementations for paper tables.
from adaptive_nesy_gen.evaluation import aggregate_results, leakage_audit
from adaptive_nesy_gen.retrieval import load_manifest
import json, pandas as pd
training_reports = [s.report for s in load_manifest(MANIFEST) if s.split == 'train']
rows = []
for name, path in ablation_files.items():
    records = [json.loads(line) for line in open(path) if line.strip()]
    summary = aggregate_results(records)
    summary.update(leakage_audit([r['final_report'] for r in records], training_reports))
    rows.append({'ablation': name, **summary})
summary_df = pd.DataFrame(rows).set_index('ablation')
summary_path = f'{OUTPUT_ROOT}/ablation_summary.csv'
summary_df.to_csv(summary_path)
display(summary_df)
print('Saved:', summary_path)


## Publication checklist

- Run the full official test split; never expose its references before inference finishes.
- Record indexing, retrieval, generation, claim-verification and end-to-end latency; graph calls/report; escalation over all and linked claims; peak GPU memory; and index size.
- Run official BLEU-1–4, ROUGE-L, METEOR, CIDEr, CheXbert/CheXpert, and RadGraph F1.
- Evaluate the linker separately and audit exact/high-overlap leakage, same-study exclusion, prediction frequency and diversity.
- Use paired bootstrap confidence intervals and expert review before claiming hallucination reduction.
- Treat LTN scores as implemented constraint satisfaction and traces as procedural, not complete causal explanations.